In [1]:
%pip install -q youtube-transcript-api langchain-community langchain-openai faiss-cpu tiktoken python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

d:\AgenticAI\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\iampr\AppData\Local\Temp\ipykernel_13020\3829014579.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## Step 1a - Indexing (Document Ingestion)

In [40]:
video_id = "Gfr50f6ZBvo"

# Create an instance
ytt_api = YouTubeTranscriptApi()

# Fetch the transcript (returns a FetchedTranscript object)
fetched_transcript = ytt_api.fetch(video_id, languages=['en'])

# Convert to the old dictionary format (if needed)
transcript_list = fetched_transcript.to_raw_data()

# Flatten to plain text
transcript = " ".join(chunk["text"] for chunk in transcript_list)
print(f"Successfully retrieved transcript with {len(transcript)} characters")

Successfully retrieved transcript with 133836 characters


In [42]:
transcript_list

[{'text': 'the following is a conversation with',
  'start': 0.08,
  'duration': 3.44},
 {'text': 'demus hasabis', 'start': 1.76, 'duration': 4.96},
 {'text': 'ceo and co-founder of deepmind', 'start': 3.52, 'duration': 5.119},
 {'text': 'a company that has published and builds',
  'start': 6.72,
  'duration': 4.48},
 {'text': 'some of the most incredible artificial',
  'start': 8.639,
  'duration': 4.561},
 {'text': 'intelligence systems in the history of',
  'start': 11.2,
  'duration': 4.8},
 {'text': 'computing including alfred zero that',
  'start': 13.2,
  'duration': 3.68},
 {'text': 'learned', 'start': 16.0, 'duration': 2.96},
 {'text': 'all by itself to play the game of gold',
  'start': 16.88,
  'duration': 4.559},
 {'text': 'better than any human in the world and',
  'start': 18.96,
  'duration': 5.6},
 {'text': 'alpha fold two that solved protein',
  'start': 21.439,
  'duration': 4.241},
 {'text': 'folding', 'start': 24.56, 'duration': 4.16},
 {'text': 'both tasks consider

## Step 1b - Indexing (Text Splitting)

In [12]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [13]:
len(chunks)

168

In [14]:
chunks[100]

Document(metadata={}, page_content="and and kind of come up with descriptions of the electron clouds where they're gonna go how they're gonna interact when you put two elements together uh and what we try to do is learn a simulation uh uh learner functional that will describe more chemistry types of chemistry so um until now you know you can run expensive simulations but then you can only simulate very small uh molecules very simple molecules we would like to simulate large materials um and so uh today there's no way of doing that and we're building up towards uh building functionals that approximate schrodinger's equation and then allow you to describe uh what the electrons are doing and all materials sort of science and material properties are governed by the electrons and and how they interact so have a good summarization of the simulation through the functional um but one that is still close to what the actual simulation would come out with so what um how difficult is that to ask w

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [15]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)

In [16]:
vector_store.index_to_docstore_id

{0: 'ad6777aa-b5aa-4cc2-9f1b-d9585716c804',
 1: '37c9884b-2917-44ee-9b34-911e26a7b6b3',
 2: '117be91f-aab6-48e9-906e-7a74670435ca',
 3: 'efe7a6fc-414b-4701-9479-01b7c8a719bc',
 4: '0ef2c86d-1cc6-463a-abd0-bcb35389d64d',
 5: '1cb36a43-17da-4015-81e8-40e16717b897',
 6: 'bc51ef0b-1b1d-424b-9421-c4aea29cc598',
 7: 'f598a712-033f-4dc8-a459-e01fee32450f',
 8: '7a76a142-b6df-41c5-a081-f05962ebcffe',
 9: '03a34a60-3153-432b-bff1-e71b9ca809fe',
 10: 'edf6c829-0967-4bc0-bcbb-004488a9402e',
 11: 'acfc235e-3160-45c9-8851-8495d0609734',
 12: 'f5e5b2ed-0aa7-4a11-8dc5-668428acc9ba',
 13: '688fd2ad-bc81-413f-9356-ad1b9389ea0f',
 14: '19e525d4-f974-41e9-a516-245afe8ef0dc',
 15: '45135a0d-e61e-4c74-af1a-91dd0d4c5acb',
 16: '261b3224-7a11-4a4d-b6b5-25efc4bbbb11',
 17: 'd14b3757-062b-4a79-bf8d-6913bca0cac6',
 18: 'e76f44fc-1964-4bf0-a650-7ce57c0f6f17',
 19: 'a2fe3d99-01fc-4887-ad8d-adcc191be502',
 20: '511b4fa7-0509-4498-9a3f-c9235666644d',
 21: '721833e7-c89c-4439-87c1-c2faf9c553dd',
 22: '3aa205d4-38d0-

In [18]:
vector_store.get_by_ids(['14ea0a97-f325-4e15-8dad-f5f31b8b765a'])

[Document(id='14ea0a97-f325-4e15-8dad-f5f31b8b765a', metadata={}, page_content="are repeatable um it feels like it's almost structured in a way to be conducive to gaining knowledge so i feel like and you know why should computers be even possible isn't that amazing that uh computational electronic devices can can can can be possible and they're made of sand our most you know common element that we have you know silicon that on the on the earth's crust they could be made of diamond or something then we would have only had one computer yeah right so it's a lot of things are kind of slightly suspicious to me it sure as heck sounds this puzzle sure sounds like something we talked about earlier what it takes to to design a game that's really fun to play for prolonged periods of time and it does seem like this puzzle like you mentioned the more you learn about it the more you realize how little you know so it humbles you but excites you by the possibility of learning more it's one heck of a 

## Step 2 - Retrieval

In [19]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [20]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000169EA1CEC80>, search_kwargs={'k': 4})

In [21]:
retriever.invoke('What is deepmind')

[Document(id='ad6777aa-b5aa-4cc2-9f1b-d9585716c804', metadata={}, page_content="the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal qu

## Step 3 - Augmentation

In [22]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

In [23]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [24]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed in very short 1 line"
retrieved_docs    = retriever.invoke(question)

In [25]:
retrieved_docs

[Document(id='cd01d1c7-2df2-45d8-b948-ecf958b0b76f', metadata={}, page_content="so we with this problem and we published it in a nature paper last year uh we held the fusion that we held the plasma in specific shapes so actually it's almost like carving the plasma into different shapes and control and hold it there for the record amount of time so um so that's one of the problems of of fusion sort of um solved so i have a controller that's able to no matter the shape uh contain it continue yeah contain it and hold it in structure and there's different shapes that are better for for the energy productions called droplets and and and so on so um so that was huge and now we're looking we're talking to lots of fusion startups to see what's the next problem we can tackle uh in the fusion area so another fascinating place in a paper title pushing the frontiers of density functionals by solving the fractional electron problem so you're taking on modeling and simulating the quantum mechanical 

In [26]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"so we with this problem and we published it in a nature paper last year uh we held the fusion that we held the plasma in specific shapes so actually it's almost like carving the plasma into different shapes and control and hold it there for the record amount of time so um so that's one of the problems of of fusion sort of um solved so i have a controller that's able to no matter the shape uh contain it continue yeah contain it and hold it in structure and there's different shapes that are better for for the energy productions called droplets and and and so on so um so that was huge and now we're looking we're talking to lots of fusion startups to see what's the next problem we can tackle uh in the fusion area so another fascinating place in a paper title pushing the frontiers of density functionals by solving the fractional electron problem so you're taking on modeling and simulating the quantum mechanical behavior of electrons yes um can you explain this work and can ai model and\n\n

In [27]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [29]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      so we with this problem and we published it in a nature paper last year uh we held the fusion that we held the plasma in specific shapes so actually it's almost like carving the plasma into different shapes and control and hold it there for the record amount of time so um so that's one of the problems of of fusion sort of um solved so i have a controller that's able to no matter the shape uh contain it continue yeah contain it and hold it in structure and there's different shapes that are better for for the energy productions called droplets and and and so on so um so that was huge and now we're looking we're talking to lots of fusion startups to see what's the next problem we can tackle uh in the fusion area so another fascinating place in a paper title pushing the frontiers of density functionals

## Step 4 - Generation

In [30]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, the topic of nuclear fusion is discussed, focusing on using AI and deep reinforcement learning to control high-temperature plasmas and solve challenges in fusion energy.


## Building a Chain

In [31]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [32]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
#this chain creates embeddings of question and perofrm semantic search on vector store to extarct most relevant documents and create a context
#finally we will have context and question

parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [34]:
parallel_chain.invoke('who is Demis')

{'context': "the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get

In [43]:
#passing this context & question to next steps in chain

In [35]:
parser = StrOutputParser()

In [36]:
main_chain = parallel_chain | prompt | llm | parser

In [37]:
main_chain.invoke('Can you summarize the video in 1 line wihtout missing any important detail')

'The conversation explores the interplay between science and engineering in solving intelligence, emphasizing the need for deeper explanations in physics and the challenges of specifying high-level abstract concepts for AI systems.'